# 04 · Modelos paramétricos AFT / Accelerated failure time models

$$\log T = \beta^\top x + \sigma\,\varepsilon$$

Em vez de multiplicar o *risco*, as covariáveis **aceleram ou desaceleram o tempo**:
$\exp(\beta_j)$ é um **time ratio**; TR = 1,2 significa permanência 20% **maior**.
Vantagem: não assume riscos proporcionais e permite extrapolar a curva.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))  # works even without `pip install -e .`
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from turnover_survival import config, data, nonparametric, models, evaluation

sns.set_theme(style="whitegrid")
pd.set_option("display.float_format", "{:.3f}".format)
config.FIGURES_DIR.mkdir(parents=True, exist_ok=True)
T, E = config.DURATION_COL, config.EVENT_COL

df = data.load_clean()
print(f"{len(df)} employees | {df[E].sum()} left | {1 - df[E].mean():.1%} censored")

In [ ]:
from lifelines import WeibullFitter, LogNormalFitter, LogLogisticFitter, ExponentialFitter
from lifelines.plotting import qq_plot

X = data.make_design_matrix(df)
train, test = data.split(X)

## Qual distribuição? / Which distribution? (univariate)

In [ ]:
fitters = [ExponentialFitter(), WeibullFitter(), LogNormalFitter(), LogLogisticFitter()]
fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))
rows = []
for f, ax in zip(fitters, axes):
    f.fit(df[T], df[E])
    qq_plot(f, ax=ax)
    ax.set_title(f.__class__.__name__.replace("Fitter", ""))
    rows.append({"distribution": f.__class__.__name__.replace("Fitter", ""), "AIC": f.AIC_})
fig.tight_layout()
fig.savefig(config.FIGURES_DIR / "04_qq.png", dpi=120, bbox_inches="tight")
pd.DataFrame(rows).sort_values("AIC")

Pontos sobre a diagonal no QQ-plot indicam bom ajuste da distribuição.

## AFT com covariáveis / AFT regression

In [ ]:
aft = models.fit_aft_models(train)
models.compare_aft(aft)

In [ ]:
best_name = models.compare_aft(aft).iloc[0]["model"]
best = aft[best_name]
print("Best by AIC:", best_name)

s = best.summary.loc[best.summary.index.get_level_values(0).str.contains("lambda_|mu_|alpha_", regex=True)]
s = s.reset_index().query("covariate != 'Intercept'")
s["time_ratio"] = np.exp(s["coef"])
s[["covariate", "time_ratio", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]].sort_values("time_ratio")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 10))
best.plot(ax=ax)
ax.set_title(f"{best_name} AFT coefficients (log time ratio)")
fig.savefig(config.FIGURES_DIR / "04_aft_coefs.png", dpi=120, bbox_inches="tight")

**Leitura:** time ratio < 1 → a característica **encurta** o tempo até a saída (maior risco);
> 1 → **alonga** a permanência.

## Mediana prevista no teste / Predicted median tenure

In [ ]:
pred = best.predict_median(models.features(test)).rename("predicted_median_months")
pd.concat([test[[T, E]], pred], axis=1).head(10)

### Conclusões / Takeaways
> _Preencha após rodar com os dados reais. / Fill in after running on the real data._